In [1]:
import ee
from pathlib import Path
import shutil
import subprocess
import time

In [2]:
ee.Initialize(project='eecc-maureen')

#### Step 1. Data Extraction and Export (JRC GSW seasonality)

In [3]:
# Porto Alegre bounding box [minLon, minLat, maxLon, maxLat]
poa = ee.Geometry.Rectangle([-51.30344, -30.26945, -51.018852, -29.932474])

# JRC Global Surface Water seasonality (months water present, 0-12)
gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
seasonality = gsw.select("seasonality")

img = (seasonality
       .clip(poa)
       .reproject(crs="EPSG:3857", scale=30)
       .toByte())

task = ee.batch.Export.image.toDrive(
    image=img,
    description="poa_gsw_seasonality_30m_v1_4",
    folder="gee_exports",
    fileNamePrefix="poa_gsw_seasonality_30m_v1_4",
    region=poa,
    scale=30,
    crs="EPSG:3857",
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

task.start()
print("Started Drive export:", task.id)

Started Drive export: 3COKRLNDZ5GMXPN3YRB2OXIS


In [ ]:
# Optional: wait for task completion
# while task.active():
#     print("Status:", task.status().get("state"))
#     time.sleep(10)
# print(task.status())

#### Step 2. Convert to COG and Generate Tiles

In [ ]:
base = Path("data")
in_tif = base / "poa_gsw_seasonality_30m_v1_4.tif"
out_dir = Path("out/gsw_seasonality_v1_4")
cog_tif = out_dir / "poa_gsw_seasonality_30m_v1_4_cog.tif"
visual_tiles_dir = out_dir / "tiles_visual"
value_tiles_dir = out_dir / "tiles_values"

out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Convert to COG
subprocess.run([
    "gdal_translate", str(in_tif), str(cog_tif),
    "-of", "COG",
    "-ot", "Byte",
    "-co", "COMPRESS=DEFLATE",
    "-co", "RESAMPLING=NEAREST",
    "-co", "OVERVIEWS=AUTO"
], check=True)
print("Created COG:", cog_tif)

# Visual tiles
colorized_tif = out_dir / "poa_gsw_seasonality_colorized.tif"
colors_txt = base / "gsw_seasonality_colors.txt"
subprocess.run([
    "gdaldem", "color-relief", str(cog_tif), str(colors_txt), str(colorized_tif)
], check=True)

# Preflight gdal2tiles runtime
gdal2tiles = shutil.which("gdal2tiles.py")
if not gdal2tiles:
    raise RuntimeError("gdal2tiles.py not found in PATH. Install GDAL first (e.g., `brew install gdal`).")

gdal2tiles_python = None
with open(gdal2tiles, "r", encoding="utf-8", errors="ignore") as f:
    first_line = f.readline().strip()
if first_line.startswith("#!"):
    parts = first_line[2:].split()
    if parts:
        if parts[0].endswith("env") and len(parts) > 1:
            gdal2tiles_python = shutil.which(parts[1])
        else:
            gdal2tiles_python = parts[0]
if not gdal2tiles_python:
    gdal2tiles_python = shutil.which("python3") or shutil.which("python")
if not gdal2tiles_python:
    raise RuntimeError("Could not determine Python interpreter for gdal2tiles.py")
subprocess.run([gdal2tiles_python, "-c", "import numpy"], check=True, capture_output=True)

visual_tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none",
    str(colorized_tif), str(visual_tiles_dir)
], check=True)
print("Visual tiles written to:", visual_tiles_dir)

# Value tiles: keep seasonality months in pixel value (R channel)
value_encoded_tif = out_dir / "poa_gsw_seasonality_value_encoded.tif"
subprocess.run([
    "gdal_translate", str(cog_tif), str(value_encoded_tif), "-ot", "Byte"
], check=True)

value_tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none",
    str(value_encoded_tif), str(value_tiles_dir)
], check=True)
print("Value tiles written to:", value_tiles_dir)
print("Decode in app with: seasonality_months = R")